In [20]:
!pip install -q google-generativeai

In [23]:
!pip install -q -U bitsandbytes accelerate

In [11]:
from huggingface_hub import login

# You will be prompted to enter your Hugging Face token.
# You can find your token on the Hugging Face website: huggingface.co/settings/tokens
login()

In [7]:
import os
import re
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
# Import specific modules for 4-bit quantization
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from sklearn.model_selection import train_test_split
# Ensure you have bitsandbytes and accelerate installed for 4-bit loading
import bitsandbytes # required for load_in_4bit
import accelerate # required for device_map="auto"

# ---------------- Configuration ----------------
# NOTE: Cannot use "gemini-2.5-flash-lite". Replaced with a compatible,
# high-quality open-source model for QLoRA fine-tuning.
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
TUNE_MODEL_PATH = "./model/mistral_text_to_sql_lora.pt"
# Hugging Face dataset id for CUB-200 (Caltech-UCSD Birds)
HF_BIRD_DATASET = "sisinflab-ai/GradeSQL-training-dataset-bird-unbalanced"
MAX_LENGTH = 512

# Global tokenizer variable for map function
tokenizer = None

# ---------------- 1. Data Preprocessing (load HF bird dataset & synthesize text->SQL) ----------------
def create_text_to_sql_data_from_hf(sample_limit=None, random_seed=42):
    """
    Loads bird dataset from Hugging Face and creates synthetic instruction->SQL pairs.
    sample_limit: if set, limits number of examples (useful for quick tests).
    """
    print("Loading bird dataset from Hugging Face:", HF_BIRD_DATASET)
    try:
        # Attempt to load only the training part
        ds = load_dataset(HF_BIRD_DATASET, split="train")
    except Exception as e:
        print(f"Error loading 'train' split, attempting to load all: {e}")
        ds_dict = load_dataset(HF_BIRD_DATASET)
        ds = ds_dict['train'] if 'train' in ds_dict else next(iter(ds_dict.values()))

    print("Dataset loaded. Examples:", len(ds))

    def get_species_name(example):
        # Try common keys for species/class names
        for key in ("species", "label_name", "name", "class", "label_text", "text"):
            if key in example and example[key]:
                return str(example[key]).split(",")[0].strip()
        if "label" in example:
            return f"SpeciesID_{example['label']}"
        return f"Unknown_{hash(str(example)) % 1000}"

    # Build small set of distinct species names
    species_names = []
    for ex in ds:
        nm = get_species_name(ex)
        species_names.append(nm)
    species_names = list(pd.unique(species_names))

    # For simplicity create synthetic question/sql template pairs
    data = []
    n = len(species_names) if sample_limit == None else min(sample_limit, len(species_names))
    rng = np.random.default_rng(seed=random_seed)
    size_to_select = min(n, len(species_names))
    selected = rng.choice(species_names, size=size_to_select, replace=False)

    # Template for SQL-Tuning: The model generates the 'completion' after the prompt.
    INSTRUCTION_PROMPT = "### Instruction: Generate a SQL query for the following question about the Bird table.\n"
    for name in selected:
        # Example templates
        templates = [
            (f"What is the wingspan of the '{name}'?", f"SELECT Wingspan_cm FROM Bird WHERE Species = '{name}';"),
            (f"Is the '{name}' listed as 'Endangered'?", f"SELECT Species FROM Bird WHERE Species = '{name}' AND Status = 'Endangered';"),
            (f"Which habitat does '{name}' prefer?", f"SELECT Habitat FROM Bird WHERE Species = '{name}';"),
            (f"Count species in the 'Forest' habitat that include '{name}' genus (example).",
              "SELECT COUNT(Species) FROM Bird WHERE Habitat = 'Forest';"),
        ]
        q, sql = rng.choice(templates)
        data.append({"prompt": q, "completion": sql})

    df = pd.DataFrame(data)
    # This template is crucial for instruction-tuning
    df['text'] = df.apply(lambda row: f"{INSTRUCTION_PROMPT}### Question: {row['prompt']}\n### SQL: {row['completion']}", axis=1)

    # Split into train/eval
    train_df, eval_df = train_test_split(df[['text']], test_size=0.2, random_state=random_seed)
    train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
    eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))
    return train_dataset, eval_dataset

# ---------------- Tokenization helper ----------------
def tokenize_function(examples):
    # The global tokenizer is now used here
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

# ---------------- Fine-tune (Q-LoRA) ----------------
def fine_tune_model(train_dataset, eval_dataset):
    print(f"\n--- Starting Fine-Tuning of {MODEL_NAME} ---")
    global tokenizer

    # -- tokenizer --
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # -- model loading (with QLoRA config) --
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            load_in_4bit=True,      # QLoRA configuration
            device_map="auto"       # Use accelerate to manage device placement
        )
        print("Loaded model with load_in_4bit=True (QLoRA)")
    except Exception as e:
        print(f"Warning: load_in_4bit failed or not supported. Falling back to fp16 full-load. Error: {e}")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            device_map="auto"
        )

    # Crucial for QLoRA performance: enable gradient checkpointing and prepare model
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    # -- LoRA config --
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        # Target modules for Mistral/Gemma type architectures
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # -- tokenise datasets --
    tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

    # collator that pads variable length sequences
    data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        optim="paged_adamw_32bit",
        save_strategy="epoch",
        logging_dir='./logs',
        logging_steps=10,
        learning_rate=2e-4,
        fp16=True, # Recommended when using 4-bit loading
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        lr_scheduler_type="constant",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_eval_dataset,
        data_collator=data_collator
    )

    print("\nStarting training...")
    trainer.train()

    tune_model_dir = os.path.dirname(TUNE_MODEL_PATH)
    os.makedirs(tune_model_dir, exist_ok=True)
    # Save LoRA adapter weights (PEFT standard)
    trainer.model.save_pretrained(tune_model_dir)
    print(f"\n✅ Fine-tuned model saved as LoRA weights to: {tune_model_dir}")

    # Additionally, save the PEFT adapter state_dict to a single .pt file as requested.
    # trainer.model is a PeftModel, its state_dict() by default contains only the trainable adapter weights.
    torch.save(trainer.model.state_dict(), TUNE_MODEL_PATH)
    print(f"✅ Fine-tuned LoRA adapter weights also saved to single file: {TUNE_MODEL_PATH}")

# ---------------- Prediction & SQL token accuracy ----------------
def measure_sql_accuracy(model_dir, dataset):
    print(f"\n--- Loading Model from {model_dir} for Prediction and Accuracy ---")

    # 1. Load the base model with the same configuration (quantization)
    try:
        base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto", load_in_4bit=True)
    except Exception:
        base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

    base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if base_tokenizer.pad_token is None:
        base_tokenizer.pad_token = base_tokenizer.eos_token
    base_tokenizer.padding_side = "left" # Set padding side to 'left' for inference

    # 2. Load the PEFT model structure and adapter weights
    model = PeftModel.from_pretrained(base_model, model_dir)

    model.eval()
    if torch.cuda.is_available():
        model.to("cuda")

    all_predictions = []
    all_references = []
    INSTRUCTION_PROMPT = "### Instruction: Generate a SQL query for the following question about the Bird table.\n"

    def extract_sql_and_predict(row):
        # parse text block for question + reference SQL
        match = re.search(r"### Question: (.*?)\n### SQL: (.*)", row['text'], re.DOTALL)
        if not match:
            print(f"Skipping malformed example: {row['text'][:50]}...")
            return
        question, reference_sql = match.groups()
        # Normalize reference SQL for comparison
        normalized_ref = reference_sql.strip().upper().replace(";", "").replace("(", " ").replace(")", " ").split()
        all_references.append(normalized_ref)

        # The prompt is everything up to '### SQL:'
        prompt_text = f"{INSTRUCTION_PROMPT}### Question: {question}\n### SQL:"
        input_enc = base_tokenizer(prompt_text, return_tensors="pt", truncation=True).to(model.device)

        with torch.no_grad():
            out = model.generate(
                **input_enc,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=base_tokenizer.eos_token_id,
                temperature=0.0
            )

        # Decode and extract generated SQL
        full_output = base_tokenizer.decode(out[0], skip_special_tokens=True)

        # Get the part *after* the prompt text
        generated_text = full_output[len(prompt_text):].strip()

        # Simple extraction of the first SQL statement generated
        pred_sql_match = re.search(r"(SELECT.*?);", generated_text, re.DOTALL | re.IGNORECASE)
        pred_sql = pred_sql_match.group(0) if pred_sql_match else generated_text.split("###")[0]

        # Normalize predicted SQL for comparison
        normalized_pred = pred_sql.strip().upper().replace(";", "").replace("(", " ").replace(")", " ").split()
        all_predictions.append(normalized_pred)
        print(f"\nQ: {question}\nRef: {reference_sql}\nPred: {pred_sql}")

    # Run on the evaluation dataset
    for item in dataset:
        extract_sql_and_predict(item)

    # compute token-level accuracy
    total_tokens = 0
    correct_tokens = 0
    for ref_tokens, pred_tokens in zip(all_references, all_predictions):
        total_tokens += len(ref_tokens)
        min_len = min(len(ref_tokens), len(pred_tokens))
        for i in range(min_len):
            if ref_tokens[i] == pred_tokens[i]:
                correct_tokens += 1
    token_accuracy = (correct_tokens / total_tokens) if total_tokens > 0 else 0

    print("\n" + "="*50)
    print(f"SQL Token Accuracy: {token_accuracy:.4f} (Matched / Reference tokens)")
    print(f"Total Reference Tokens: {total_tokens}")
    print(f"Correctly Matched Tokens: {correct_tokens}")
    print("="*50)

def main():
    # GPU check
    if not torch.cuda.is_available():
        print("Warning: CUDA GPU not found. You can still run on CPU but it will be very slow for LLM training.")

    # 1. create datasets
    train_set, eval_set = create_text_to_sql_data_from_hf(sample_limit=200)

    # 2. fine tune
    tune_model_dir = os.path.dirname(TUNE_MODEL_PATH)
    fine_tune_model(train_set, eval_set)

    # 3. evaluate
    # Check for the presence of the adapter model file
    if os.path.exists(os.path.join(tune_model_dir, "adapter_model.safetensors")) or os.path.exists(os.path.join(tune_model_dir, "adapter_model.bin")):
        measure_sql_accuracy(tune_model_dir, eval_set)
    else:
        print("Fine-tuned model adapter file not found. Skipping evaluation.")

# ---------------- Main ----------------
if __name__ == "__main__":
    main()

Loading bird dataset from Hugging Face: sisinflab-ai/GradeSQL-training-dataset-bird-unbalanced
Dataset loaded. Examples: 82640


/tmp/ipython-input-1537599296.py:58: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  species_names = list(pd.unique(species_names))



--- Starting Fine-Tuning of mistralai/Mistral-7B-v0.1 ---


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...


✅ Fine-tuned model saved as LoRA weights to: ./model
✅ Fine-tuned LoRA adapter weights also saved to single file: ./model/mistral_text_to_sql_lora.pt

--- Loading Model from ./model for Prediction and Accuracy ---


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



Q: Is the 'SpeciesID_1' listed as 'Endangered'?
Ref: SELECT Species FROM Bird WHERE Species = 'SpeciesID_1' AND Status = 'Endangered';
Pred: SELECT Species FROM Bird WHERE Species = 'SpeciesID_1' AND Endangered = 'Endangered';

SQL Token Accuracy: 0.9167 (Matched / Reference tokens)
Total Reference Tokens: 12
Correctly Matched Tokens: 11


In [6]:
!ls -F ./model

adapter_config.json  adapter_model.safetensors	README.md
